## Faster R-CNN extension: Faster R-CNN ResNet-50 FPN v2
The goal is to investigate whether the improved Faster R-CNN ResNet-50 FPN v2 architecture achieves better object detection performance than the original Faster R-CNN ResNet-50 FPN v1 on the same data and with the same training settings.

The effect of scale augmentation on model performance is also evaluated.

It is still a two-stage anchor-based object detector (RPN + RoI head), using an existing torchvision implementation and COCO-pretrained weights.

What changed in v2:

- ResNet-50 backbone trained with a newer training recipe.

- Batch normalization added to the FPN and RPN heads.

- The RoI head uses four 3×3 convolutional layers and one fully connected layer instead of two fully connected layers.

V2: Only the model architecture is changed.

V2 + scale augmentation: During training, the shorter side of the image is randomly selected between 576 and 1024 pixels. For 20% of frames, an additional downscaling of 0.35–0.7× is applied first. This is because the test set contains frames with heights of 382 and 540 pixels, which are smaller than any training frames. Evaluation always uses the original 720/1280 resize settings.

In [1]:
import json
import sys
import pandas as pd
from IPython.display import display

import faster_rcnn_v2
import train_pilot
import train_all_folds

pd.set_option('display.width', 200)
METRICS = ['mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR']
V1_NAME = 'lr_scheduler (v1)'

In [2]:
v1_result = json.loads(train_pilot.REFERENCE.read_text())
v1_official_all = pd.read_csv(faster_rcnn_v2.ROOT / 'faster_rcnn/official_val_folds.csv').query("run == 'lr_scheduler'")
v1_official = v1_official_all.set_index('fold').loc[0]

In [3]:
v2_results = {exp['name']: faster_rcnn_v2.run_experiment(exp, faster_rcnn_v2.PILOT_FOLD) for exp in faster_rcnn_v2.EXPERIMENTS}
v2_official = {exp['name']: train_pilot.official_validation(exp['name']) for exp in faster_rcnn_v2.EXPERIMENTS}

v2_lr_scheduler/fold0 : result already exists - skipping training
v2_lr_scheduler_scale_aug/fold0 : result already exists - skipping training


In [4]:
configs = {V1_NAME: v1_result['cfg'], **{name: result['cfg'] for name, result in v2_results.items()}}
setting_names = ['epochs', 'bs', 'lr', 'wd', 'warmup', 'scheduler', 'min_size', 'max_size', 'rpn_pre_train', 'rpn_post_train',
                 'rpn_pre_test', 'rpn_post_test', 'rpn_batch', 'roi_batch', 'roi_pos_frac', 'box_loss', 'ignore_ioa',
                 'trainable_layers', 'seed', 'train_min_sizes', 'train_max_size', 'p_down', 'down_range']
display(pd.DataFrame({run: {setting: config.get(setting, '-') for setting in setting_names} for run, config in configs.items()}))

,lr_scheduler (v1),v2_lr_scheduler,v2_lr_scheduler_scale_aug
epochs,6,6,6
bs,4,4,4
lr,0.01,0.01,0.01
wd,0.0001,0.0001,0.0001
warmup,300,300,300
scheduler,multistep,multistep,multistep
min_size,720,720,720
max_size,1280,1280,1280
rpn_pre_train,2000,2000,2000
rpn_post_train,2000,2000,2000


In [5]:
parameters = {'v1': v1_result['n_params_M'], **{name: result['n_params_M'] for name, result in v2_results.items()}}
print('parameters (M):', {name: round(value, 1) for name, value in parameters.items()})

parameters (M): {'v1': 41.3, 'v2_lr_scheduler': 43.3, 'v2_lr_scheduler_scale_aug': 43.3}


### Fold 0 (Model Selection) and Official Validation (Verification)

In [6]:
rows = [dict(run=V1_NAME, best_epoch=v1_result['best_epoch'], **{m: v1_result['metrics'][m] for m in METRICS},
             ms_per_img=v1_result['ms_per_img'], official_mAP=v1_official.mAP, official_AP50=v1_official.AP50,
             official_APs=v1_official.APs)]
for name, result in v2_results.items():
    official = v2_official[name]['metrics']
    rows.append(dict(run=name, best_epoch=result['best_epoch'], **{m: result['metrics'][m] for m in METRICS},
                     ms_per_img=result['ms_per_img'], official_mAP=official['mAP'], official_AP50=official['AP50'],
                     official_APs=official['APs']))
fold0_table = pd.DataFrame(rows).set_index('run')
fold0_table['dmAP_fold0'] = fold0_table.mAP - fold0_table.mAP.iloc[0]
fold0_table['dmAP_official'] = fold0_table.official_mAP - fold0_table.official_mAP.iloc[0]
display(fold0_table.round(4))

,best_epoch,mAP,AP50,AP75,APs,APm,APl,AR,ms_per_img,official_mAP,official_AP50,official_APs,dmAP_fold0,dmAP_official
run,,,,,,,,,,,,,,
lr_scheduler (v1),5,0.2377,0.4370,0.2382,0.1030,0.1945,0.3684,0.3761,27.0977,0.1960,0.4003,0.0918,0.0000,0.0000
v2_lr_scheduler,5,0.2672,0.4903,0.2627,0.1153,0.2361,0.3818,0.3879,25.4724,0.2123,0.4242,0.0979,0.0295,0.0163
v2_lr_scheduler_scale_aug,1,0.2812,0.5239,0.2726,0.1233,0.2427,0.4067,0.4418,28.5884,0.2187,0.4498,0.0958,0.0435,0.0227


In [7]:
best_v2 = fold0_table.mAP.iloc[1:].max()
continue_to_5_folds = best_v2 >= v1_result['metrics']['mAP'] + train_pilot.MARGIN
print('decision:', 'continue with folds 1-4' if continue_to_5_folds else 'stop: gain below the fixed margin of 0.010')

decision: continue with folds 1-4


### mAP after every epoch (fold 0)

In [8]:
epochs = pd.Index(range(1, 7), name='epoch')
curves = pd.DataFrame({'lr_scheduler (v1)': [epoch['mAP'] for epoch in v1_result['history']],
                       **{name: [epoch['mAP'] for epoch in result['history']] for name, result in v2_results.items()}},
                      index=epochs)
display(curves.round(4))

,lr_scheduler (v1),v2_lr_scheduler,v2_lr_scheduler_scale_aug
epoch,,,
1,0.2121,0.2630,0.2812
2,0.2154,0.2515,0.2807
3,0.2260,0.2392,0.2673
4,0.2326,0.2637,0.2681
5,0.2377,0.2672,0.2731
6,0.2368,0.2667,0.2738


In [9]:
minutes_per_epoch = pd.DataFrame({f'{name} (min/epoch)': [epoch['epoch_seconds'] / 60 for epoch in result['history']]
                                  for name, result in v2_results.items()}, index=epochs)
display(minutes_per_epoch.round(1))

,v2_lr_scheduler (min/epoch),v2_lr_scheduler_scale_aug (min/epoch)
epoch,,
1,7.0,8.2
2,6.6,5.9
3,6.5,10.4
4,6.6,10.6
5,6.9,10.6
6,6.9,10.5


### Per-class AP (fold 0)

In [10]:
per_class = {'lr_scheduler (v1)': v1_result['metrics']['per_class'],
             **{name: result['metrics']['per_class'] for name, result in v2_results.items()}}
display(pd.DataFrame(per_class).round(4))

,lr_scheduler (v1),v2_lr_scheduler,v2_lr_scheduler_scale_aug
person,0.1756,0.2021,0.2101
bicycle,0.0524,0.0640,0.0744
car,0.4897,0.4983,0.5128
van,0.2702,0.2926,0.3192
truck,0.1711,0.1871,0.2165
tricycle,0.2514,0.2629,0.2685
bus,0.4350,0.5494,0.5658
motor,0.0563,0.0809,0.0824


### All 5 folds

In [11]:
chosen_experiment, _ = train_all_folds.chosen()
chosen_name = chosen_experiment['name']
five_folds = pd.read_csv(faster_rcnn_v2.HERE / f'{chosen_name}_5fold.csv')
five_folds['v1_official_mAP'] = five_folds.fold.map(v1_official_all.set_index('fold').mAP)
five_folds['dCV'] = five_folds.cv_mAP - five_folds.v1_cv_mAP
five_folds['dOfficial'] = five_folds.official_mAP - five_folds.v1_official_mAP
display(five_folds.round(4))

,fold,cv_mAP,v1_cv_mAP,official_mAP,best_epoch,v1_official_mAP,dCV,dOfficial
0,0,0.2812,0.2377,0.2187,1,0.1960,0.0435,0.0227
1,1,0.2649,0.2208,0.2222,1,0.1858,0.0441,0.0364
2,2,0.2545,0.2380,0.2219,4,0.1962,0.0164,0.0256
3,3,0.2377,0.2015,0.2155,4,0.1828,0.0362,0.0327
4,4,0.1801,0.1571,0.2312,5,0.1935,0.0230,0.0376


In [12]:
print('mean held-out mAP: v2', round(five_folds.cv_mAP.mean(), 4), '| v1', round(five_folds.v1_cv_mAP.mean(), 4),
      '| folds improved', int((five_folds.dCV > 0).sum()), 'of 5')
print('mean official-val mAP: v2', round(five_folds.official_mAP.mean(), 4), '| v1', round(five_folds.v1_official_mAP.mean(), 4),
      '| folds improved', int((five_folds.dOfficial > 0).sum()), 'of 5')

mean held-out mAP: v2 0.2437 | v1 0.211 | folds improved 5 of 5
mean official-val mAP: v2 0.2219 | v1 0.1909 | folds improved 5 of 5


In [13]:
histories = {fold: json.loads((faster_rcnn_v2.OUT / chosen_name / f'fold{fold}/result.json').read_text())['history'] for fold in range(5)}
fold_curves = pd.DataFrame({f'fold {fold}': [epoch['mAP'] for epoch in history] for fold, history in histories.items()}, index=epochs)
display(fold_curves.round(4))

,fold 0,fold 1,fold 2,fold 3,fold 4
epoch,,,,,
1,0.2812,0.2649,0.2496,0.2236,0.1530
2,0.2807,0.2483,0.2365,0.2351,0.1653
3,0.2673,0.2554,0.2418,0.2212,0.1562
4,0.2681,0.2548,0.2545,0.2377,0.1798
5,0.2731,0.2632,0.2470,0.2290,0.1801
6,0.2738,0.2607,0.2521,0.2316,0.1771


### Comparing v2 With and Without Scale Augmentation

The `v2_lr_scheduler` experiment uses Faster R-CNN v2 with the same training settings as Model 1, but without scale augmentation. All 5 folds are trained using the same code, data splits, and seed (`train_without_scale_aug.py`).

The goal is to separately evaluate the impact of the v2 architecture and scale augmentation. Each best checkpoint is evaluated once on the official validation set.

In [14]:
no_augmentation = pd.read_csv(faster_rcnn_v2.HERE / 'v2_lr_scheduler_5fold.csv').set_index('fold')
with_augmentation = five_folds.set_index('fold')
steps = pd.DataFrame({
    'v1 CV': with_augmentation.v1_cv_mAP, 'v2 CV': no_augmentation.cv_mAP, 'v2 + scale aug CV': with_augmentation.cv_mAP,
    'v1 official': with_augmentation.v1_official_mAP, 'v2 official': no_augmentation.official_mAP,
    'v2 + scale aug official': with_augmentation.official_mAP})
display(steps.round(4))

,v1 CV,v2 CV,v2 + scale aug CV,v1 official,v2 official,v2 + scale aug official
fold,,,,,,
0,0.2377,0.2672,0.2812,0.1960,0.2123,0.2187
1,0.2208,0.2465,0.2649,0.1858,0.1966,0.2222
2,0.2380,0.2394,0.2545,0.1962,0.2137,0.2219
3,0.2015,0.2354,0.2377,0.1828,0.2135,0.2155
4,0.1571,0.1676,0.1801,0.1935,0.2183,0.2312


In [15]:
for split in ['CV', 'official']:
    architecture = steps[f'v2 {split}'] - steps[f'v1 {split}']
    augmentation = steps[f'v2 + scale aug {split}'] - steps[f'v2 {split}']
    print(f'{split}: architecture v1 -> v2 {architecture.mean():+.4f} (folds better {int((architecture > 0).sum())}/5) | '
          f'scale augmentation {augmentation.mean():+.4f} (folds better {int((augmentation > 0).sum())}/5)')

CV: architecture v1 -> v2 +0.0202 (folds better 5/5) | scale augmentation +0.0124 (folds better 5/5)
official: architecture v1 -> v2 +0.0200 (folds better 5/5) | scale augmentation +0.0110 (folds better 5/5)


### Kaggle Pipelines with Faster R-CNN v2 (`ensemble/`)

The two Faster R-CNN Kaggle pipelines were tested with five v2 checkpoints. All other settings remained the same, including TTA, WBF, FCOS fusion, temporal processing, parameter search, and evaluation.

- `ensemble/predict_v2_views.py`: Generates six TTA predictions for each checkpoint on the tuning, validation, and test sets. The results are checked against the original v1 inference code.
- `ensemble/ensemble_pipeline.py`: Tunes parameters using only out-of-fold training frames (56 videos, 80 consecutive frames each), then evaluates the results once on official validation.
- **Pipeline A:** Faster R-CNN v2 + TTA + WBF + temporal processing.
- **Pipeline B:** Pipeline A with FCOS fusion added before temporal processing.

In [16]:
ENSEMBLE_DIR = faster_rcnn_v2.ROOT / 'final_pipeline' / 'ensemble'
sys.path.insert(0, str(ENSEMBLE_DIR))
import ensemble_pipeline

tuning_columns = ['key', 'mAP', 'AP50', 'APs', 'APm', 'APl'] + [f'mAP_fold{k}' for k in range(5)]
for stage in ['v2_temporal', 'v2_fusion', 'v2_fusion_temporal']:
    stage_table = pd.read_csv(ENSEMBLE_DIR / 'tuning' / f'{stage}.csv').sort_values('mAP', ascending=False)
    print(stage, '(tuning frames, top 5 of', len(stage_table), ')')
    display(stage_table[tuning_columns].head(5).round(4))

v2_temporal (tuning frames, top 5 of 19 )


,key,mAP,AP50,APs,APm,APl,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
6,s_high 0.2 alpha 0.8 beta 0.6,0.2887,0.5366,0.0929,0.2532,0.5403,0.3193,0.3171,0.2983,0.3193,0.2260
4,s_high 0.2 alpha 0.65 beta 0.6,0.2886,0.5358,0.0925,0.2526,0.5394,0.3189,0.3155,0.2983,0.3184,0.2253
11,s_high 0.3 alpha 0.8 beta 0.6,0.2885,0.5360,0.0925,0.2530,0.5395,0.3175,0.3171,0.2974,0.3184,0.2267
7,s_high 0.2 alpha 0.8 beta 1.0,0.2884,0.5361,0.0929,0.2522,0.5400,0.3191,0.3170,0.2982,0.3189,0.2243
1,Faster R-CNN parameters (frozen),0.2884,0.5352,0.0923,0.2525,0.5384,0.3172,0.3154,0.2977,0.3177,0.2259


v2_fusion (tuning frames, top 5 of 14 )


,key,mAP,AP50,APs,APm,APl,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
5,w_fcos 0.5 iou 0.65,0.2976,0.5478,0.1051,0.2607,0.5380,0.3293,0.3206,0.3179,0.3256,0.2205
7,w_fcos 0.75 iou 0.65,0.2975,0.5477,0.1061,0.2616,0.5356,0.3306,0.3218,0.3167,0.3260,0.2194
9,w_fcos 1.0 iou 0.65,0.2964,0.5464,0.1065,0.2614,0.5307,0.3312,0.3216,0.3148,0.3257,0.2182
3,w_fcos 0.25 iou 0.65,0.2952,0.5445,0.1021,0.2572,0.5372,0.3251,0.3163,0.3182,0.3232,0.2217
13,w_fcos 0.75 iou 0.55 fcos score^1.5,0.2952,0.5460,0.1056,0.2597,0.5335,0.3282,0.3192,0.3151,0.3246,0.2162


v2_fusion_temporal (tuning frames, top 5 of 19 )


,key,mAP,AP50,APs,APm,APl,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
7,s_high 0.2 alpha 0.8 beta 1.0,0.3050,0.5636,0.1077,0.2702,0.5463,0.3407,0.3339,0.3238,0.3293,0.2270
6,s_high 0.2 alpha 0.8 beta 0.6,0.3049,0.5635,0.1079,0.2702,0.5473,0.3400,0.3339,0.3238,0.3291,0.2282
5,s_high 0.2 alpha 0.65 beta 1.0,0.3047,0.5631,0.1077,0.2694,0.5456,0.3402,0.3318,0.3233,0.3316,0.2266
11,s_high 0.3 alpha 0.8 beta 0.6,0.3047,0.5630,0.1075,0.2707,0.5449,0.3391,0.3345,0.3231,0.3291,0.2333
4,s_high 0.2 alpha 0.65 beta 0.6,0.3047,0.5631,0.1078,0.2694,0.5466,0.3396,0.3317,0.3233,0.3314,0.2278


In [17]:
print('selected on the tuning frames:', json.loads((ENSEMBLE_DIR / 'tuning/selected.json').read_text()))

selected on the tuning frames: {'v2_temporal': {'s_high': 0.2, 's_low': 0.01, 'iou_link': 0.5, 'max_gap': 2, 'min_len': 3, 'alpha': 0.8, 'beta': 0.6, 'gamma': 0.75}, 'fusion': {'weight_fcos': 0.5, 'iou': 0.65}, 'fusion_power': 1.0, 'fusion_temporal': {'s_high': 0.2, 's_low': 0.01, 'iou_link': 0.5, 'max_gap': 2, 'min_len': 3, 'alpha': 0.8, 'beta': 1.0, 'gamma': 0.75}}


In [18]:
official_results = pd.read_csv(ENSEMBLE_DIR / 'official_validation_results.csv')
video_columns = [column for column in official_results.columns if column.startswith('mAP_video_')]
display(official_results[['key'] + METRICS + video_columns].round(4))
display(pd.DataFrame(ensemble_pipeline.decision(official_results)).T)

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_video_uav0000086_00000_v,mAP_video_uav0000117_02622_v,mAP_video_uav0000137_00458_v,mAP_video_uav0000182_00000_v,mAP_video_uav0000268_05773_v,mAP_video_uav0000305_00000_v,mAP_video_uav0000339_00001_v
0,v1: Faster R-CNN 5 checkpoints TTA (0.2587 pip...,0.2364,0.4771,0.2068,0.1267,0.2221,0.3994,0.4727,0.1610,0.2915,0.3066,0.2037,0.1810,0.3194,0.2785
1,v1: Faster R-CNN + temporal (0.2810 pipeline),0.2450,0.4987,0.2124,0.1315,0.2277,0.4217,0.4731,0.1787,0.3037,0.3156,0.2144,0.1844,0.3258,0.2969
2,v1: Faster R-CNN + FCOS fusion,0.2446,0.4854,0.2166,0.1280,0.2340,0.4072,0.4955,0.1805,0.2890,0.3138,0.2087,0.1855,0.3644,0.2908
3,v1: Faster R-CNN + FCOS fusion + temporal (tuned),0.2528,0.5044,0.2219,0.1405,0.2392,0.4230,0.4959,0.1926,0.3006,0.3200,0.2197,0.1881,0.3699,0.2947
4,v2 5 checkpoints TTA,0.2717,0.5299,0.2464,0.1334,0.2517,0.4579,0.4994,0.3067,0.3082,0.3573,0.2295,0.1571,0.3369,0.2987
5,v2 + temporal (Faster R-CNN parameters),0.2837,0.5546,0.2552,0.1542,0.2590,0.4819,0.4996,0.3163,0.3219,0.3593,0.2362,0.1594,0.3554,0.3160
6,v2 + FCOS fusion,0.2748,0.5285,0.2532,0.1378,0.2597,0.4569,0.5109,0.2998,0.3088,0.3551,0.2319,0.1862,0.3761,0.3157
7,A: v2 + temporal (tuned),0.2841,0.5573,0.2545,0.1560,0.2584,0.4894,0.4996,0.3185,0.3279,0.3603,0.2384,0.1612,0.3553,0.3187
8,B: v2 + FCOS fusion + temporal (tuned),0.2928,0.5588,0.2692,0.1660,0.2659,0.5005,0.5113,0.3034,0.3357,0.3588,0.2415,0.1922,0.3808,0.3404


,mAP,gain,videos_improved,videos,adopt
A: v2 + temporal (tuned),0.284104,0.0313,5,7,True
B: v2 + FCOS fusion + temporal (tuned),0.29281,0.040006,7,7,True


In [ ]:
report = {'submissions': {}}
for report_name in ['faster_rcnn_updated_submission_report.json', 'faster_rcnn_updated_fcos_submission_report.json']:
    report['submissions'].update(json.loads((faster_rcnn_v2.ROOT / 'kaggle_submissions' / report_name).read_text())['submissions'])

for name, submission in report['submissions'].items():
    print(name, '->', submission['submission'])
    print('   checks:', 'all pass' if all(submission['checks'].values()) else submission['checks'])
    stats = submission['stats']
    print('   stats:', {name: stats[name] for name in ['rows', 'detections', 'max_per_image', 'mean_per_image', 'classes_present']})

A: v2 + temporal (tuned) -> E:\ucu_hw2\final_pipeline\ensemble\faster_rcnn_v2_5fold_tta_wbf_temporal_submission.csv
   checks: all pass
   stats: {'rows': 3811, 'detections': 1589981, 'max_per_image': 500, 'mean_per_image': 417.20834426659667, 'classes_present': [0, 1, 2, 3, 4, 5, 6, 7]}
B: v2 + FCOS fusion + temporal (tuned) -> E:\ucu_hw2\final_pipeline\ensemble\faster_rcnn_v2_fcos_5fold_tta_wbf_fusion_temporal_submission.csv
   checks: all pass
   stats: {'rows': 3811, 'detections': 1905500, 'max_per_image': 500, 'mean_per_image': 500.0, 'classes_present': [0, 1, 2, 3, 4, 5, 6, 7]}
